In [ ]:
import pandas as pd
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pytz
import requests
import re
import psycopg2

from functions import read_db_credentials, connect_to_db, load_json_data_from_db_as_json, save_df_to_db, data_from_data_sink

In [ ]:
# def read_db_credentials(path="data/config.txt"):
#     creds = {}
#     with open(path, "r") as f:
#         for line in f:
#             key, value = line.strip().split("=")
#             creds[key] = value
#     return creds


# def connect_to_db(creds):
#     return psycopg2.connect(
#         host=creds["host"],
#         port=creds["port"],
#         dbname=creds["database"],
#         user=creds["user"],
#         password=creds["password"]
#     )
# def data_from_data_sink(query):
#     creds = read_db_credentials()
#     conn = connect_to_db(creds)
#     cur = conn.cursor()
    
#     df = pd.read_sql_query(query, conn)
    
#     conn.close()
    
#     return df

# def save_df_to_db(df, table_name, conflict_column='user_number'):
#     creds = read_db_credentials()
#     conn = connect_to_db(creds)
#     cursor = conn.cursor()
    
#     df = df.where(pd.notnull(df), None)  # Replace NaN with None for SQL

#     columns = list(df.columns)
#     column_names = ', '.join(columns)
#     placeholders = ', '.join(['%s'] * len(columns))

#     # Set update expression for each column
#     update_assignments = ', '.join([f"{col} = EXCLUDED.{col}" for col in columns if col != conflict_column])

#     insert_query = f"""
#         INSERT INTO {table_name} ({column_names})
#         VALUES ({placeholders})
#         ON CONFLICT ({conflict_column})
#         DO UPDATE SET {update_assignments};
#     """

#     for _, row in df.iterrows():
#         cursor.execute(insert_query, row.tolist())

#     conn.commit()
#     cursor.close()
#     conn.close()
#     return "saved (with upsert)"

    
# def load_json_data_from_db_as_json(user, source ):
#     creds = read_db_credentials()
#     conn = connect_to_db(creds)

#     # query = f"SELECT raw_json FROM fact_raw_data WHERE data_source = '{source}' AND user_number = {user} ;"
    
#     cursor = conn.cursor()
#     cursor.execute(query)
#     result = cursor.fetchone()
#     conn.close()
#     #return (result)
#     # # Parsen der JSON-Inhalte aus der 'data'-Spalte
#     parsed_data = result[0] if result else {}


#     # # Rückgabe als JSON-String (optional indent für Lesbarkeit)
#     return parsed_data


In [28]:
with open("data/cur_user_selected.txt", "r", encoding="utf-8") as f:
    user = int(f.read().strip())


sptfy_data = load_json_data_from_db_as_json(user, "spotify")
print(sptfy_data)

{'extracted_at': '2025-06-28T22:03:43.782525Z', 'user_profile': {'id': 'dave23042000', 'display_name': 'dave23042000', 'country': 'DE', 'product': 'premium'}, 'top_artists': [{'id': '08GQAI4eElDnROBrJRGE0X', 'name': 'Fleetwood Mac', 'genres': ['classic rock', 'yacht rock', 'soft rock'], 'popularity': 85, 'followers': 13335667, 'spotify_url': 'https://open.spotify.com/artist/08GQAI4eElDnROBrJRGE0X'}, {'id': '3nwKjjo8rZtE0rRVgz4OPB', 'name': 'S.T.S', 'genres': ['schlager', 'neue deutsche welle', 'schlagerparty'], 'popularity': 54, 'followers': 251338, 'spotify_url': 'https://open.spotify.com/artist/3nwKjjo8rZtE0rRVgz4OPB'}, {'id': '4vWQjpI68kWBNkXOEbi1D6', 'name': 'Rainhard Fendrich', 'genres': ['neue deutsche welle', 'schlager', 'schlagerparty', 'german pop'], 'popularity': 52, 'followers': 244433, 'spotify_url': 'https://open.spotify.com/artist/4vWQjpI68kWBNkXOEbi1D6'}, {'id': '0WwSkZ7LtFUFjGjMZBMt6T', 'name': 'Dire Straits', 'genres': ['classic rock'], 'popularity': 81, 'followers': 8

In [29]:
# Top-Level Keys anzeigen
data = sptfy_data
import json

def print_json_structure(data, indent=0):
    spacer = "  " * indent
    if isinstance(data, dict):
        for key, value in data.items():
            print(f"{spacer}\"{key}\": ", end="")
            if isinstance(value, (dict, list)):
                print()
                print_json_structure(value, indent + 1)
            else:
                print(type(value).__name__)
    elif isinstance(data, list):
        print(f"{spacer}[")
        if data:
            print_json_structure(data[0], indent + 1)
        else:
            print(f"{'  ' * (indent + 1)}<empty>")
        print(f"{spacer}]")
    else:
        print(f"{spacer}{type(data).__name__}")

# JSON-Datei laden

# Struktur ausgeben
print_json_structure(data)


"extracted_at": str
"user_profile": 
  "id": str
  "display_name": str
  "country": str
  "product": str
"top_artists": 
  [
    "id": str
    "name": str
    "genres": 
      [
        str
      ]
    "popularity": int
    "followers": int
    "spotify_url": str
  ]
"recently_played": 
  [
    "id": str
    "name": str
    "popularity": int
    "duration_ms": int
    "explicit": bool
    "spotify_url": str
    "artists": 
      [
        "id": str
        "name": str
      ]
    "album": 
      "id": str
      "name": str
      "release_date": str
    "played_at": str
    "play_count": int
  ]
"collaborative_playlists": 
  [
    <empty>
  ]


In [30]:
from collections import Counter
import requests

artist_genre_cache = {}  # Cache für Artist-Genres

def get_weighted_genres(track_query, target_artist):
    if target_artist.lower() in artist_genre_cache:
        return artist_genre_cache[target_artist.lower()]

    access_token = auth.get_token()
    headers = {"Authorization": f"Bearer {access_token}"}
    params = {"q": track_query, "type": "track", "limit": 20}
    response = requests.get("https://api.spotify.com/v1/search", headers=headers, params=params)
    data = response.json()

    genre_counter = Counter()
    total_artist_matches = 0

    for track in data.get("tracks", {}).get("items", []):
        matched_artist = next(
            (artist for artist in track["artists"] if artist["name"].lower() == target_artist.lower()), None
        )
        if matched_artist:
            total_artist_matches += 1
            artist_id = matched_artist["id"]
            artist_data = requests.get(f"https://api.spotify.com/v1/artists/{artist_id}", headers=headers).json()
            genres = artist_data.get("genres", [])
            genre_counter.update(genres)

    if total_artist_matches == 0 or not genre_counter:
        artist_genre_cache[target_artist.lower()] = {}
        return {}

    total_genre_mentions = sum(genre_counter.values())
    normalized_genres = {
        genre: round(count / total_genre_mentions, 3)
        for genre, count in genre_counter.items()
    }

    artist_genre_cache[target_artist.lower()] = normalized_genres
    return normalized_genres




In [31]:
import pandas as pd

def extract_recent_tracks(data: dict) -> pd.DataFrame:
    """
    Extracts track name, artist name, and play duration from a user's recently played Spotify data.

    :param data: Dictionary structured as described (with 'recently_played' list of tracks)
    :return: DataFrame with columns: track_name, artist_name, duration_ms
    """
    recently_played = data.get("recently_played", [])

    if not recently_played:
        return pd.DataFrame(columns=["track_name", "artist_name", "duration_ms"])

    rows = []
    for track in recently_played[:500]:  # Limit to first 500
        track_name = track.get("name", "NA")
        duration_ms = track.get("duration_ms", 0)
        artist_info = track.get("artists", [])
        timestamp = track.get("played_at", [])
        artist_name = artist_info[0].get("name") if artist_info else "Unknown"
        
        rows.append({
            "track_name": track_name,
            "artist_name": artist_name,
            "duration_ms": duration_ms,
            "timestamp": timestamp
        })

    return pd.DataFrame(rows)


df = extract_recent_tracks(sptfy_data)
print(df)

                                      track_name          artist_name  \
0                                    Wahre Liebe      Virginia Jetzt!   
1           (I Can't Get No) Satisfaction - Mono   The Rolling Stones   
2                                Fire Water Burn      Bloodhound Gang   
3                                    Tanto amore             Udo West   
4                           All The Small Things            blink-182   
5   Another One Bites The Dust - Remastered 2011                Queen   
6                            I Get Around (Mono)       The Beach Boys   
7                                Beautiful World            Colin Hay   
8                                   Baba O'Riley              The Who   
9                                 Boys Don't Cry             The Cure   
10                                   Fäuste hoch        Irie Révoltés   
11                                    Chandelier                  Sia   
12              Can't Hold Us (feat. Ray Dalton)   

In [32]:

from data.secrets import auth, client_secret
df["weightedGenres"] = df.apply(
    lambda row: get_weighted_genres(row['track_name'], row['artist_name']),
    axis=1
)

In [33]:
import requests
import base64

client_id = '8541e8dfc0fd4a86916d0d98cdb150ad'
client_secret = 'cade2256b5804831a11349f86e6a1784'

import requests
import base64
import time

class SpotifyAuth:
    def __init__(self, client_id, client_secret):
        self.client_id = client_id
        self.client_secret = client_secret
        self.access_token = None
        self.token_expires_at = 0  # Unix timestamp

    def _fetch_token(self):
        auth_str = f"{self.client_id}:{self.client_secret}"
        b64_auth_str = base64.b64encode(auth_str.encode()).decode()

        response = requests.post(
            "https://accounts.spotify.com/api/token",
            data={"grant_type": "client_credentials"},
            headers={"Authorization": f"Basic {b64_auth_str}"}
        )

        if response.status_code == 200:
            token_data = response.json()
            self.access_token = token_data["access_token"]
            # Spotify gibt Gültigkeit in Sekunden an (meist 3600)
            self.token_expires_at = time.time() + token_data.get("expires_in", 3600) - 60
        else:
            raise Exception(f"Fehler beim Abrufen des Tokens: {response.status_code} - {response.text}")


    def get_token(self):
        if not self.access_token or time.time() >= self.token_expires_at:
            self._fetch_token()
        return self.access_token
    

auth = SpotifyAuth(client_id = '8541e8dfc0fd4a86916d0d98cdb150ad',
client_secret = 'cade2256b5804831a11349f86e6a1784')



In [34]:
import pandas as pd
import re


# Regex-basierte Mapping-Regeln (von spezifisch zu allgemein)
REGEX_GENRE_TO_USER_TYPE = {
    # HIPHOP / RAP
    r".*\bhip hop\b.*": "hiphop_head",
    r".*\bgrime\b.*": "hiphop_head",
    r".*\bdrill\b.*": "hiphop_head",
    r".*\brap\b.*": "hiphop_head",
    r".*\burbaine\b.*": "hiphop_head",
    r".*\btrap\b.*": "hiphop_head",
    r".*\bgrime\b.*": "hiphop_head",
    r".*\buk grime\b.*": "hiphop_head",

    # ELECTRONIC
    r".*\bhouse\b.*": "electronic_addict",
    r".*\btrance\b.*": "electronic_addict",
    r".*\btechno\b.*": "electronic_addict",
    r".*\bbig room\b.*": "electronic_addict",
    r".*\bedm\b.*": "electronic_addict",
    r".*\belectronica\b.*": "electronic_addict",
    r".*\bdance\b.*": "electronic_addict",
    r".*\bdrum and bass\b.*": "electronic_addict",
    r".*\bfuture house\b.*": "electronic_addict",
    r".*\bprogressive house\b.*": "electronic_addict",
    r".*\btropical house\b.*": "electronic_addict",

    # POP
    r".*\bpop\b.*": "pop_lover",
    r".*\bsoft pop\b.*": "pop_lover",
    r".*\bart pop\b.*": "pop_lover",
    r".*\bindie pop\b.*": "pop_lover",
    r".*\bmodern pop\b.*": "pop_lover",
    r".*\bmoroccan pop\b.*": "pop_lover",
    r".*\begyptian pop\b.*": "pop_lover",
    r".*\bturkish pop\b.*": "pop_lover",
    r".*\blatin pop\b.*": "pop_lover",
    r".*\bk-pop\b.*": "pop_lover",
    r".*\bt-pop\b.*": "pop_lover",

    # INDIE / ALTERNATIVE
    r".*\bindie rock\b.*": "indie_explorer",
    r".*\balternative rock\b.*": "indie_explorer",
    r".*\bindie\b.*": "indie_explorer",
    r".*\bsinger-songwriter\b.*": "indie_explorer",
    r".*\bitalian singer-songwriter\b.*": "indie_explorer",

    # JAZZ
    r".*\bjazz\b.*": "jazz_purist",
    r".*\bfrench jazz\b.*": "jazz_purist",

    # CLASSICAL / MUSICAL
    r".*\bopera\b.*": "classical_connoisseur",
    r".*\bmusicals\b.*": "classical_connoisseur",
    r".*\bnueva trova\b.*": "classical_connoisseur",

    # ROCK
    r".*\brock\b.*": "rock_head",
    r".*\bgarage rock\b.*": "rock_head",

    # FUNK / SOUL / R&B
    r".*\br&b\b.*": "funk_soul_groover",
    r".*\bfrench r&b\b.*": "funk_soul_groover",
    r".*\bsoul\b.*": "funk_soul_groover",

    # COUNTRY / FOLK
    r".*\bfolk\b.*": "country_heart",
    r".*\bfolk pop\b.*": "country_heart",
    r".*\bmariachi\b.*": "country_heart",
    r".*\btrova\b.*": "country_heart",
    r".*\bbolero\b.*": "country_heart",

    # GLOBAL
    r".*\bafro.*": "global_vibes_fan",
    r".*\bdembow\b.*": "global_vibes_fan",
    r".*\blatin\b.*": "global_vibes_fan",
    r".*\bmoroccan rap\b.*": "global_vibes_fan",
    r".*\bkhaleeji\b.*": "global_vibes_fan",
    r".*\barabic hip hop\b.*": "global_vibes_fan",
    r".*\bvariet[eé] fran[aç]aise\b.*": "global_vibes_fan",
    r".*\bchanson\b.*": "global_vibes_fan",
    r".*\bitalo dance\b.*": "global_vibes_fan",
    r".*\barabesk\b.*": "global_vibes_fan",
    r".*\bra[iï]\b.*": "global_vibes_fan",
}


# Funktion zum Zuordnen via Regex
def map_genre_to_user_type(genre):
    for pattern, user_type in REGEX_GENRE_TO_USER_TYPE.items():
        if re.search(pattern, genre, re.IGNORECASE):
            return user_type
    return "mix_consumer"




In [35]:
rows = []
for _, row in df.iterrows():
    genres_dict = row.get("weightedGenres", {})
    if isinstance(genres_dict, dict):
        for genre, weight in genres_dict.items():
            rows.append({
                "trackName": row['track_name'],
                "artistName": row['artist_name'],
                "genre": genre,
                "weight": weight
            })

df_genres = pd.DataFrame(rows)
print(df_genres)
genre_summary = df_genres.groupby("genre", as_index=False)["weight"].sum().sort_values(by="weight", ascending=False)
genre_summary["user_type"] = genre_summary["genre"].apply(map_genre_to_user_type)


genre_summary["user_type"] = genre_summary["genre"].apply(map_genre_to_user_type)
print(genre_summary["genre"])
# Gruppieren und Gewicht aufsummieren
result = genre_summary.groupby("user_type", as_index=False)["weight"].sum()
total_weight = result ["weight"].sum()
result["normalized_weight"] = (result["weight"] / total_weight)*100
result_genre = result.sort_values("normalized_weight", ascending=False)
display(result.sort_values("normalized_weight", ascending=False))

                                       trackName          artistName  \
0           (I Can't Get No) Satisfaction - Mono  The Rolling Stones   
1           (I Can't Get No) Satisfaction - Mono  The Rolling Stones   
2                                    Tanto amore            Udo West   
3                                    Tanto amore            Udo West   
4                           All The Small Things           blink-182   
5                           All The Small Things           blink-182   
6                           All The Small Things           blink-182   
7                           All The Small Things           blink-182   
8                           All The Small Things           blink-182   
9   Another One Bites The Dust - Remastered 2011               Queen   
10  Another One Bites The Dust - Remastered 2011               Queen   
11  Another One Bites The Dust - Remastered 2011               Queen   
12                           I Get Around (Mono)      The Beach 

,user_type,weight,normalized_weight
4,rock_head,3.449,43.117890
2,mix_consumer,1.850,23.127891
3,pop_lover,1.200,15.001875
0,electronic_addict,1.000,12.501563
1,indie_explorer,0.500,6.250781


In [36]:
import pandas as pd
from datetime import datetime
print(df)

# Zeitstempel umwandeln
df["timestamp"]= pd.to_datetime(df["timestamp"])

# Zeitstempel umwandeln
df["timestamp"]= pd.to_datetime(df["timestamp"])

# Minuten berechnen
df['minutes'] = df['duration_ms'] / 1000 / 60

# Datum extrahieren
df['date'] = df["timestamp"].dt.date

# Wochentag ermitteln: Montag = 0, Sonntag = 6
df['weekday'] = df["timestamp"].dt.weekday
df['is_weekend'] = df['weekday'] >= 5  # Samstag=5, Sonntag=6 → True

# Minuten pro Tag + Wochenende-Markierung
daily_minutes = df.groupby(['date', 'is_weekend'])['minutes'].sum().reset_index()

# Mittelwert berechnen
mean_per_group = daily_minutes.groupby('is_weekend')['minutes'].mean().reset_index()

# Lesbare Labels
mean_per_group['day_type'] = mean_per_group['is_weekend'].map({True: 'Weekend', False: 'Weekday'})
mean_per_group = mean_per_group[['day_type', 'minutes']]

print(daily_minutes)
print(mean_per_group)

# Tageszeit-Funktion
def get_day_part(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "midday"
    elif 17 <= hour < 23:
        return "evening"
    else:
        return "night"

# Attribute berechnen
df['date'] = df["timestamp"].dt.date
df['weekday'] = df["timestamp"].dt.weekday  # Monday = 0
df['is_weekend'] = df['weekday'] >= 5
df['day_part'] = df["timestamp"].dt.hour.apply(get_day_part)

# Aggregation: Minuten pro Tagtyp und Tageszeit
df['day_type'] = df['is_weekend'].map({True: 'Weekend', False: 'Weekday'})
print(df)
# Gruppenbasierte Mittelwerte berechnen
grouped = df.groupby(['date','day_type', 'day_part'])['minutes'].sum().reset_index().groupby(['day_type', 'day_part'])['minutes'].mean().reset_index()


print(grouped)


                                      track_name          artist_name  \
0                                    Wahre Liebe      Virginia Jetzt!   
1           (I Can't Get No) Satisfaction - Mono   The Rolling Stones   
2                                Fire Water Burn      Bloodhound Gang   
3                                    Tanto amore             Udo West   
4                           All The Small Things            blink-182   
5   Another One Bites The Dust - Remastered 2011                Queen   
6                            I Get Around (Mono)       The Beach Boys   
7                                Beautiful World            Colin Hay   
8                                   Baba O'Riley              The Who   
9                                 Boys Don't Cry             The Cure   
10                                   Fäuste hoch        Irie Révoltés   
11                                    Chandelier                  Sia   
12              Can't Hold Us (feat. Ray Dalton)   

In [37]:

type_mapping = []

type_mapping.append({
    "listener_type": "late_night_listener",
    "day_type": "Weekday",
    "day_part": "night",
    "weight": 1.0
})
type_mapping.append({
    "listener_type": "weekwnd_party_listener",
    "day_type": "Weekend",
    "day_part": "night",
    "weight": 1.0
})

# Weekday Focus Day
type_mapping.append({
    "listener_type": "working_focus_listener",
    "day_type": "Weekday",
    "day_part": "morning",
    "weight": 0.5
})
type_mapping.append({
    "listener_type": "working_focus_listener",
    "day_type": "Weekday",
    "day_part": "midday",
    "weight": 0.5
})

# Weekend Focus Day
type_mapping.append({
    "listener_type": "weekend_breakfast_listener",
    "day_type": "Weekend",
    "day_part": "morning",
    "weight": 0.5
})
type_mapping.append({
    "listener_type": "weekend_breakfast_listener",
    "day_type": "Weekend",
    "day_part": "midday",
    "weight": 0.5
})

# Weekday Evening
type_mapping.append({
    "listener_type": "weekday_sundown_listener",
    "day_type": "Weekday",
    "day_part": "evening",
    "weight": 1.0
})

# Weekend Evening
type_mapping.append({
    "listener_type": "weekend_prime_time_listener",
    "day_type": "Weekend",
    "day_part": "evening",
    "weight": 1.0
})

map_df = pd.DataFrame(type_mapping)
merged = pd.merge(grouped, map_df, on=["day_type", "day_part"], how="inner")
merged["weighted_minutes"] = merged["minutes"] * merged["weight"]
result = merged.groupby("listener_type")["weighted_minutes"].sum().reset_index()
result = result.sort_values(by="weighted_minutes", ascending=False).reset_index(drop=True)
total = result["weighted_minutes"].sum()
result["normalized"] = result["weighted_minutes"] / total
# Sicherstellen, dass mindestens 3 Zeilen vorhanden sind
import numpy as np
missing_rows = max(0, 3 - len(result))
if missing_rows > 0:
    empty_rows = pd.DataFrame({
        "listener_type": [np.nan] * missing_rows,
        "weighted_minutes": [np.nan] * missing_rows,
        "normalized": [np.nan] * missing_rows
    })
    result = pd.concat([result, empty_rows], ignore_index=True)

print(result)

                 listener_type  weighted_minutes  normalized
0  weekend_prime_time_listener          38.77195    0.678152
1   weekend_breakfast_listener          18.40100    0.321848
2                          NaN               NaN         NaN


| Type                                  | Description                                                                     | Rule                                                       |
| ------------------------------------- | ------------------------------------------------------------------------------- | ---------------------------------------------------------- |
| 🎧 **late\_night\_listener**          | Mostly listens at night – prefers chill, lo-fi, or introspective music.         | Night listening ≥ 40% of total time                        |
| ☀️ **weekday\_focus\_day\_listener**  | Listens primarily during mornings and midday on weekdays – productive sessions. | Morning + midday ≥ 50%, with more time on weekdays         |
| ☀️ **weekend\_focus\_day\_listener**  | Listens mainly in the morning or midday on weekends – calm and balanced mood.   | Morning + midday ≥ 50%, with more time on weekends         |
| 🌆 **weekday\_prime\_time\_streamer** | Prefers evening listening on weekdays – post-work, gym, or relaxation music.    | Evening is dominant, and weekday evening > weekend evening |
| 🌆 **weekend\_prime\_time\_streamer** | Tunes in most during weekend evenings – party vibes or social listening.        | Evening is dominant, and weekend evening > weekday evening |
| 🎛️ **eclectic\_rhythm\_fan**         | No dominant time pattern – diverse and varied music preferences.                | No strong dominance across time blocks                     |
| 🔇 **silent\_type**                   | No activity or not enough data for classification.                              | Total listening time is 0                                  |


In [38]:
result_genre

,user_type,weight,normalized_weight
4,rock_head,3.449,43.117890
2,mix_consumer,1.850,23.127891
3,pop_lover,1.200,15.001875
0,electronic_addict,1.000,12.501563
1,indie_explorer,0.500,6.250781


In [39]:

res = pd.DataFrame([{
    "user_number": user,
    "sptfy_gnr_type_1": f'{result_genre["user_type"][0]} ({round(result_genre["normalized_weight"][0],0)}%)',
    "sptfy_gnr_type_2": f'{result_genre["user_type"][1]} ({round(result_genre["normalized_weight"][1],0)}%)',
    "sptfy_gnr_type_3": f'{result_genre["user_type"][2]} ({round(result_genre["normalized_weight"][2],0)}%)',
    "sptfy_lt_type_1": f'{result["listener_type"][0]} ({round(result["normalized"][0],2)*100}%)',
    "sptfy_lt_type_2": f'{result["listener_type"][1]} ({round(result["normalized"][1],2)*100}%)',
    "sptfy_lt_type_3": f'{result["listener_type"][2]} ({round(result["normalized"][2],2)*100}%)'
   
}])
print(res)

save_df_to_db(res, "dim_spotify")

   user_number           sptfy_gnr_type_1       sptfy_gnr_type_2  \
0           10  electronic_addict (13.0%)  indie_explorer (6.0%)   

       sptfy_gnr_type_3                      sptfy_lt_type_1  \
0  mix_consumer (23.0%)  weekend_prime_time_listener (68.0%)   

                      sptfy_lt_type_2 sptfy_lt_type_3  
0  weekend_breakfast_listener (32.0%)      nan (nan%)  


'saved (with upsert)'